# Exception Handling and Fallback Chains for LLM Calls

This notebook covers production-grade resilience patterns for LLM / agent calls:

- Retry with exponential backoff (recovering from transient failures)
- Model / provider fallback chains (primary -> secondary model)
- Graceful degradation (a safe, labeled response instead of a crash)
- A brief circuit breaker (stop hammering a consistently failing dependency)
- A combined pipeline demonstrated across four scenarios: normal operation, a recovered transient failure, a fallback-triggered failure, and total-failure graceful degradation

These patterns are inspired by the "Exception Handling (Fallback)" pattern common in production agentic-systems literature (e.g. the Agentic Design Patterns anthology's chapter on fallback handling) — the ideas are re-implemented here from scratch against this repo's `helpers` package, not copied from any external source.

## What we are going to do

LLM APIs fail in the real world: rate limits, timeouts, dropped connections, a provider outage, a model deprecation. A production pipeline that just calls `llm.invoke(...)` and lets exceptions propagate will crash the whole request on any hiccup. We'll build up, layer by layer:

1. A `retry_with_backoff` decorator that retries a call a bounded number of times with exponential backoff, demonstrated against a *simulated* flaky function (so the recovery is provable without depending on a real, unreliable network call).
2. A `FallbackLLM`-style pattern that tries a primary `get_llm(...)` configuration and, if it's unavailable, transparently switches to a secondary provider/model.
3. A `safe_answer` graceful-degradation wrapper that returns a clearly-labeled degraded response (using any cached/partial context available) instead of raising.
4. A small `CircuitBreaker` class (closed / open / half-open) illustrating how to stop calling a dependency that is consistently failing, rather than retrying it on every single request.
5. A `robust_answer` pipeline that wires all four together, run against four scenarios.

In [ ]:
# ============ IMPORTS & SETUP ============
import enum
import time
from functools import wraps

from dotenv import load_dotenv

from helpers import get_llm

load_dotenv()

## 1. Retry with Exponential Backoff

Many LLM-call failures are *transient*: a rate limit that clears in a second, a momentary connection reset, a provider that's briefly overloaded. The correct response to a transient failure is not to give up immediately, and not to hammer the API in a tight loop either — it's to retry a bounded number of times with an increasing delay between attempts (exponential backoff), so the dependency gets room to recover.

We simulate a flaky call with a small helper that raises a `TransientError` on its first `N` invocations and then succeeds — this lets us prove the retry logic actually recovers, without depending on a real unreliable network call.

In [ ]:
# ============ TRANSIENT ERROR + RETRY DECORATOR ============

class TransientError(Exception):
    """Represents a retryable failure: rate limit, timeout, connection reset, etc."""
    pass


def retry_with_backoff(max_attempts: int = 4, base_delay: float = 0.5, backoff_factor: float = 2.0):
    """
    Decorator that retries a function on TransientError using exponential backoff.

    Parameters
    ----------
    max_attempts : total attempts including the first call
    base_delay : delay (seconds) before the first retry
    backoff_factor : multiplier applied to the delay after each failed attempt
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            delay = base_delay
            last_exc = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except TransientError as exc:
                    last_exc = exc
                    if attempt == max_attempts:
                        break
                    print(f"  [retry] attempt {attempt}/{max_attempts} failed ({exc}); retrying in {delay:.2f}s...")
                    time.sleep(delay)
                    delay *= backoff_factor
            raise last_exc
        return wrapper
    return decorator

In [ ]:
# ============ SIMULATED FLAKY LLM CALL ============

def make_flaky_llm_call(fail_times: int, real_answer: str = "The capital of France is Paris."):
    """
    Returns a function that raises TransientError on its first `fail_times` calls,
    then returns `real_answer` on every call after that. Simulates a rate-limited
    or momentarily-unavailable LLM endpoint without a real flaky dependency.
    """
    state = {"calls": 0}

    def flaky_call(question: str) -> str:
        state["calls"] += 1
        if state["calls"] <= fail_times:
            raise TransientError(f"simulated transient failure #{state['calls']}")
        return real_answer

    return flaky_call


# Fails twice, then succeeds on the 3rd call
flaky = make_flaky_llm_call(fail_times=2)


@retry_with_backoff(max_attempts=4, base_delay=0.3, backoff_factor=2.0)
def call_with_retry(question: str) -> str:
    return flaky(question)


result = call_with_retry("What is the capital of France?")
print("Final result:", result)

### Discussion of the Output

The first two calls raise `TransientError` and are caught by the decorator, which logs the failed attempt and sleeps with an exponentially growing delay (0.3s, then 0.6s) before retrying. On the third attempt the simulated flakiness has exhausted itself and the call succeeds — the decorator returns the result normally, and the caller never sees an exception. If the underlying function kept failing past `max_attempts`, the decorator re-raises the last exception so the caller (or a higher layer, like the fallback chain below) can react to it.

## 2. Model / Provider Fallback Chain

Retries help with transient blips, but sometimes a provider is genuinely down (an outage, a revoked key, a deprecated model). In that case the right move is to fail over to a *different* model or provider rather than keep retrying the same one.

This repo's `helpers.get_llm(*, provider=None, model=None, temperature=0, verbose=True)` factory (see `helpers/utils.py` at the repo root) already supports picking an explicit provider/model — e.g. `get_llm(provider="groq", model="openai/gpt-oss-120b")` or `get_llm(provider="openai", model="gpt-4o-mini")` — or auto-selecting platform defaults when `provider` is omitted. We build a small `FallbackLLM` wrapper around two such configurations: it calls a primary "call function" and, on any failure, transparently switches to a secondary one.

We keep the LLM construction lazy (only built on first real use) via `build_llm_call`, so this cell can be read/imported even in an environment that doesn't have every provider's API key configured — and we demonstrate the *fallback control flow* itself with mock call-functions (no network needed) right after, since that's the logic that must be provably correct.

In [ ]:
# ============ FALLBACK LLM WRAPPER ============

def build_llm_call(provider: str, model: str, temperature: float = 0):
    """
    Returns a `str -> str` call function backed by helpers.get_llm(provider=..., model=...).
    The underlying LLM is constructed lazily on first call, so simply defining a
    FallbackLLM with several of these does not require every provider's API key
    to be present until that provider is actually invoked.
    """
    holder: dict = {}

    def _call(prompt: str) -> str:
        if "llm" not in holder:
            holder["llm"] = get_llm(provider=provider, model=model, temperature=temperature, verbose=False)
        return holder["llm"].invoke(prompt).content

    return _call


class FallbackLLM:
    """
    Wraps a primary and a secondary `str -> str` call function. `invoke()` tries
    the primary first; if it raises for any reason, it logs the failure and
    transparently retries against the secondary (fallback) call function.
    """

    def __init__(self, primary_call, fallback_call, verbose: bool = True):
        self.primary_call = primary_call
        self.fallback_call = fallback_call
        self.verbose = verbose

    def invoke(self, prompt: str) -> dict:
        try:
            return {"text": self.primary_call(prompt), "source": "primary"}
        except Exception as exc:
            if self.verbose:
                print(f"  [fallback] primary failed ({type(exc).__name__}: {exc}); switching to fallback model...")
            return {"text": self.fallback_call(prompt), "source": "fallback"}


# Real wiring (illustrative — requires the relevant provider API keys to actually run):
#   primary_call = build_llm_call(provider="groq", model="openai/gpt-oss-120b")
#   fallback_call = build_llm_call(provider="openai", model="gpt-4o-mini")
#   fallback_llm = FallbackLLM(primary_call, fallback_call)
#   fallback_llm.invoke("What is the capital of France?")

In [ ]:
# ============ DEMO: FALLBACK TRIGGERED ============

def flaky_primary(prompt: str) -> str:
    raise TransientError("simulated primary provider outage")


def working_secondary(prompt: str) -> str:
    return "Paris (answered by the secondary/fallback model)."


demo_fallback = FallbackLLM(primary_call=flaky_primary, fallback_call=working_secondary)
print(demo_fallback.invoke("What is the capital of France?"))

### Discussion of the Output

`FallbackLLM.invoke()` calls `flaky_primary`, which always raises — a stand-in for a provider that is fully unavailable rather than just momentarily flaky. The wrapper catches the exception, logs which provider failed and why, and re-issues the request against `working_secondary`. The returned dict's `"source"` field (`"fallback"`) makes it explicit downstream (in logs, traces, or a UI) that the answer did not come from the primary model — this matters for cost/quality accounting in production.

## 3. Graceful Degradation

Sometimes both the primary and the fallback are unavailable — a full outage across providers, or a bug that keeps raising. At that point retrying more or failing over further won't help. The last line of defense is to **not crash the pipeline**: catch the exception and return a safe, clearly-labeled degraded response, optionally seeded with any cached or partial context you already had, rather than propagating a 500 error to the end user.

In [ ]:
# ============ GRACEFUL DEGRADATION ============

def safe_answer(question: str, llm_call, cached_context: str | None = None) -> dict:
    """
    Final safety net around a `str -> str` call function. If `llm_call` raises,
    returns a clearly labeled degraded response instead of propagating the
    exception, optionally including any cached/partial context available.
    """
    try:
        return {"answer": llm_call(question), "degraded": False}
    except Exception as exc:
        degraded_msg = (
            f"[DEGRADED RESPONSE] I'm temporarily unable to fully process this request "
            f"due to a system issue ({type(exc).__name__}). "
        )
        if cached_context:
            degraded_msg += f"Here is what I can tell you from cached information: {cached_context}"
        else:
            degraded_msg += "Please try again shortly."
        return {"answer": degraded_msg, "degraded": True, "error": str(exc)}


def always_fails(question: str) -> str:
    raise RuntimeError("every downstream provider is unavailable")


print(safe_answer(
    "What is the capital of France?",
    always_fails,
    cached_context="France's capital city is Paris (from the last successful cached answer).",
))

### Discussion of the Output

`safe_answer` never lets an exception escape: it always returns a dict with an `"answer"` field, plus a `"degraded"` flag the caller can check to decide whether to log a warning, alert an on-call, or show a banner in a UI. Because `cached_context` was supplied, the degraded message still tells the user something useful ("Paris") instead of a bare apology — this is the difference between graceful degradation and just returning an error string.

## 4. Circuit Breaker (Brief)

Retrying and failing over are per-call decisions. A **circuit breaker** operates across calls: once a dependency has failed too many times in a row, it "opens" the circuit and short-circuits further calls immediately (no wasted latency or quota hammering a dependency that's clearly down) for a cooldown period. After the cooldown, it goes **half-open** and lets a single probe call through — success closes the circuit again, failure re-opens it.

This is a minimal illustration of the three states (`CLOSED` -> `OPEN` -> `HALF_OPEN`), not a production-grade implementation (a real one would be thread-safe, track failures in a sliding window, and likely live in a shared cache for multi-process deployments).

In [ ]:
# ============ CIRCUIT BREAKER ============

class CircuitState(enum.Enum):
    CLOSED = "closed"        # normal operation, calls pass through
    OPEN = "open"            # dependency is unhealthy, calls are short-circuited
    HALF_OPEN = "half_open"  # cooldown elapsed, a single probe call is allowed through


class CircuitBreaker:
    """Minimal closed/open/half-open circuit breaker around any callable."""

    def __init__(self, failure_threshold: int = 3, cooldown_seconds: float = 5.0):
        self.failure_threshold = failure_threshold
        self.cooldown_seconds = cooldown_seconds
        self.state = CircuitState.CLOSED
        self.failure_count = 0
        self.opened_at: float | None = None

    def _maybe_half_open(self):
        if self.state == CircuitState.OPEN and self.opened_at is not None:
            if time.time() - self.opened_at >= self.cooldown_seconds:
                self.state = CircuitState.HALF_OPEN
                print("  [circuit] cooldown elapsed -> HALF_OPEN (allowing one probe call)")

    def call(self, func, *args, **kwargs):
        self._maybe_half_open()

        if self.state == CircuitState.OPEN:
            raise RuntimeError("circuit is OPEN — dependency is unhealthy, call short-circuited")

        try:
            result = func(*args, **kwargs)
        except Exception:
            self.failure_count += 1
            print(f"  [circuit] failure {self.failure_count}/{self.failure_threshold} (state={self.state.value})")
            if self.state == CircuitState.HALF_OPEN or self.failure_count >= self.failure_threshold:
                self.state = CircuitState.OPEN
                self.opened_at = time.time()
                print("  [circuit] -> OPEN (tripped)")
            raise
        else:
            if self.state == CircuitState.HALF_OPEN:
                print("  [circuit] probe succeeded -> CLOSED")
            self.state = CircuitState.CLOSED
            self.failure_count = 0
            return result

In [ ]:
# ============ DEMO: TRIPPING AND RECOVERING THE BREAKER ============

def unhealthy_dependency(x):
    raise RuntimeError("dependency down")


breaker = CircuitBreaker(failure_threshold=3, cooldown_seconds=1.0)

# Three failures in a row trip the breaker to OPEN
for i in range(3):
    try:
        breaker.call(unhealthy_dependency, i)
    except Exception as exc:
        print(f"call {i}: raised {type(exc).__name__}: {exc}")

# Next call is short-circuited immediately without touching the dependency
try:
    breaker.call(unhealthy_dependency, "short-circuited-call")
except Exception as exc:
    print(f"short-circuited call raised immediately: {exc}")

# Wait for the cooldown to elapse, then probe with a now-healthy dependency
time.sleep(1.1)
print("probe result:", breaker.call(lambda x: f"ok:{x}", "recovered"))
print("final state:", breaker.state)

### Discussion of the Output

The first three calls fail and, once `failure_count` reaches `failure_threshold`, the breaker trips to `OPEN`. The fourth call never touches `unhealthy_dependency` at all — it's rejected instantly by the breaker itself, which is the whole point: stop wasting latency/quota on a dependency that's clearly down. After the 1-second cooldown, the breaker moves to `HALF_OPEN` and allows exactly one probe call through; since that probe (a healthy lambda) succeeds, the breaker closes again.

## 5. Putting It All Together: A Combined Resilience Pipeline

Now we wire retry, circuit breaker, fallback, and graceful degradation into a single `robust_answer` pipeline and run it through four scenarios:

- **A — Normal operation:** the primary succeeds on the first try.
- **B — Recovered transient failure:** the primary fails a couple of times but the retry logic recovers it, so the fallback is never even needed.
- **C — Fallback triggered:** the primary is fully down (retries exhausted), so the pipeline transparently switches to the secondary model.
- **D — Total failure -> graceful degradation:** both primary and fallback are down, so the pipeline returns a safe, labeled degraded response instead of crashing.

In [ ]:
# ============ COMBINED PIPELINE ============

def robust_answer(question: str, primary_call, fallback_call, breaker: CircuitBreaker, cached_context: str | None = None) -> dict:
    """
    Full resilience pipeline for one Q&A call:
      1. Retry the primary call with backoff (recovers from transient blips).
      2. Route the primary call through a circuit breaker (stop hammering an
         already-unhealthy primary).
      3. If the primary path is exhausted, fall back to a secondary model.
      4. If even the fallback fails, degrade gracefully instead of raising.
    """

    @retry_with_backoff(max_attempts=3, base_delay=0.2, backoff_factor=2.0)
    def _primary_with_retry(q):
        return breaker.call(primary_call, q)

    def _combined_call(q):
        try:
            return _primary_with_retry(q)
        except Exception as primary_exc:
            print(f"  [pipeline] primary path exhausted ({type(primary_exc).__name__}); trying fallback...")
            return fallback_call(q)

    return safe_answer(question, _combined_call, cached_context=cached_context)

In [ ]:
# ============ SCENARIO A: NORMAL OPERATION ============

def unused_fallback(q):
    raise AssertionError("fallback should not be called in this scenario")


breaker_a = CircuitBreaker(failure_threshold=3, cooldown_seconds=5)
primary_ok = make_flaky_llm_call(fail_times=0, real_answer="Paris is the capital of France.")

print("=== Scenario A: normal operation ===")
print(robust_answer("What is the capital of France?", primary_ok, unused_fallback, breaker_a))

In [ ]:
# ============ SCENARIO B: RECOVERED TRANSIENT FAILURE ============

breaker_b = CircuitBreaker(failure_threshold=3, cooldown_seconds=5)
# Fails twice (within the retry budget of 3 attempts), then succeeds
primary_flaky = make_flaky_llm_call(fail_times=2, real_answer="Paris is the capital of France.")

print("=== Scenario B: transient failure recovered by retry ===")
print(robust_answer("What is the capital of France?", primary_flaky, unused_fallback, breaker_b))

In [ ]:
# ============ SCENARIO C: FALLBACK TRIGGERED ============

def always_transient_failure(q):
    raise TransientError("primary provider is down")


def secondary_ok(q):
    return "Paris (answered by the fallback model)."


breaker_c = CircuitBreaker(failure_threshold=3, cooldown_seconds=5)

print("=== Scenario C: primary exhausted -> fallback triggered ===")
print(robust_answer("What is the capital of France?", always_transient_failure, secondary_ok, breaker_c))

In [ ]:
# ============ SCENARIO D: TOTAL FAILURE -> GRACEFUL DEGRADATION ============

def always_transient_failure_2(q):
    raise TransientError("primary provider is down")


def fallback_also_failing(q):
    raise RuntimeError("fallback provider is also down")


breaker_d = CircuitBreaker(failure_threshold=3, cooldown_seconds=5)

print("=== Scenario D: total failure -> graceful degradation ===")
print(robust_answer(
    "What is the capital of France?",
    always_transient_failure_2,
    fallback_also_failing,
    breaker_d,
    cached_context="France's capital city is Paris (from the last successful cached answer).",
))

### Discussion of the Output

- **Scenario A** returns `{"answer": "Paris is the capital of France.", "degraded": False}` on the very first attempt — no retries, no fallback, nothing logged.
- **Scenario B** logs two retry attempts (the backoff sleeping briefly between them) and then succeeds on the third attempt, still through the primary — the fallback function is never invoked.
- **Scenario C** exhausts all 3 retry attempts against the primary (each trips the circuit breaker's failure counter too), logs that the primary path is exhausted, and then calls the fallback, which succeeds — the response is still non-degraded, but came from the secondary model.
- **Scenario D** exhausts the primary the same way, but the fallback also raises — `safe_answer` catches that final exception and returns a `"degraded": True` response that still surfaces the cached answer ("Paris") instead of crashing or returning a raw traceback to the caller.

Wiring it this way means a single unreliable dependency, or even a full multi-provider outage, degrades the quality of one answer instead of taking down the whole pipeline.

## Summary

Key takeaways from this notebook:

- **Retry with backoff** handles transient failures (rate limits, brief timeouts) by giving the dependency room to recover, without retrying forever.
- **Fallback chains** built on `helpers.get_llm(provider=..., model=...)` let a pipeline switch to a secondary provider/model when the primary is genuinely unavailable, rather than failing the whole request.
- **Graceful degradation** is the last line of defense: when every path fails, return a clearly-labeled, still-useful response (using cached/partial context where available) instead of propagating an exception.
- **Circuit breakers** protect a consistently-failing dependency (and your own latency/quota) by short-circuiting calls during a cooldown window, rather than retrying every single request against something that is clearly down.
- Combining all four into one pipeline (`robust_answer`) demonstrates that resilience is layered: retry absorbs blips, the breaker prevents pile-on against a dead dependency, fallback covers a full provider outage, and degradation is the guaranteed-safe floor underneath all of it.

This notebook establishes a new **`Reliability_and_Fallbacks/`** track under `12_Production_and_Observability/LLMOps_and_AI_Infrastructure/`, sibling to the existing `Caching_and_Performance/`, `Cost_Monitoring/`, and `Tracing_and_Observability/` tracks.